In [1]:
# import all Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier

Load  The Dataset

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
df = pd.read_csv("/content/drive/MyDrive/loan_data.csv")

In [4]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


# **Data Understanding & Cleaning**

In [5]:
df.shape

(45000, 14)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 14 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   person_age                      45000 non-null  float64
 1   person_gender                   45000 non-null  object 
 2   person_education                45000 non-null  object 
 3   person_income                   45000 non-null  float64
 4   person_emp_exp                  45000 non-null  int64  
 5   person_home_ownership           45000 non-null  object 
 6   loan_amnt                       45000 non-null  float64
 7   loan_intent                     45000 non-null  object 
 8   loan_int_rate                   45000 non-null  float64
 9   loan_percent_income             45000 non-null  float64
 10  cb_person_cred_hist_length      45000 non-null  float64
 11  credit_score                    45000 non-null  int64  
 12  previous_loan_defaults_on_file  

In [7]:
df.describe()

,person_age,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,loan_status
count,45000.000000,4.500000e+04,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000,45000.000000
mean,27.764178,8.031905e+04,5.410333,9583.157556,11.006606,0.139725,5.867489,632.608756,0.222222
std,6.045108,8.042250e+04,6.063532,6314.886691,2.978808,0.087212,3.879702,50.435865,0.415744
min,20.000000,8.000000e+03,0.000000,500.000000,5.420000,0.000000,2.000000,390.000000,0.000000
25%,24.000000,4.720400e+04,1.000000,5000.000000,8.590000,0.070000,3.000000,601.000000,0.000000
50%,26.000000,6.704800e+04,4.000000,8000.000000,11.010000,0.120000,4.000000,640.000000,0.000000
75%,30.000000,9.578925e+04,8.000000,12237.250000,12.990000,0.190000,8.000000,670.000000,0.000000
max,144.000000,7.200766e+06,125.000000,35000.000000,20.000000,0.660000,30.000000,850.000000,1.000000


In [8]:
# Checking null Values
df.isnull().sum()

,0
person_age,0
person_gender,0
person_education,0
person_income,0
person_emp_exp,0
person_home_ownership,0
loan_amnt,0
loan_intent,0
loan_int_rate,0
loan_percent_income,0


In [9]:
# checking duplicate values
df.duplicated().sum()

np.int64(0)

In [10]:
df.columns

Index(['person_age', 'person_gender', 'person_education', 'person_income',
       'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent',
       'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length',
       'credit_score', 'previous_loan_defaults_on_file', 'loan_status'],
      dtype='object')

Checking Uniques Values Of the Columns

In [11]:
df['person_education'].unique()

array(['Master', 'High School', 'Bachelor', 'Associate', 'Doctorate'],
      dtype=object)

In [12]:
df['person_gender'].unique()

array(['female', 'male'], dtype=object)

In [13]:
df['loan_intent'].unique()

array(['PERSONAL', 'EDUCATION', 'MEDICAL', 'VENTURE', 'HOMEIMPROVEMENT',
       'DEBTCONSOLIDATION'], dtype=object)

# **Data Preprocessing**

In [14]:
# ordinal encoding
education_order = {
    "High School" : 0,
    "Associate" : 1,
    "Bachelor" : 2,
    "Master" : 3,
    "Doctorate" : 4
}
df["person_education"] = df["person_education"].map(education_order)

In [15]:
df.head()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,3,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,0,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,0,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,2,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,3,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


# **Split X & y**

In [16]:
x = df.drop("loan_status",axis=1)
y = df["loan_status"]

In [17]:
x

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file
0,22.0,female,3,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No
1,21.0,female,0,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes
2,25.0,female,0,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No
3,23.0,female,2,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No
4,24.0,male,3,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,27.0,male,1,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No
44996,37.0,female,1,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No
44997,33.0,male,1,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No
44998,29.0,male,2,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No


In [18]:
y

,loan_status
0,1
1,0
2,1
3,1
4,1
...,...
44995,1
44996,1
44997,1
44998,1


Columns Transformer

In [19]:
numeric_features = [
    "person_age","person_income",
    "person_emp_exp",
    "loan_amnt","loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score",
    "person_education"
]

categorical_features = [
    "person_gender",
    "person_home_ownership",
    "loan_intent",
    "previous_loan_defaults_on_file"
]
preprocessor = ColumnTransformer([
    ("num",StandardScaler(),numeric_features),
    ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features)
])

In [20]:
preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['person_age', 'person_income',
                                  'person_emp_exp', 'loan_amnt',
                                  'loan_int_rate', 'loan_percent_income',
                                  'cb_person_cred_hist_length', 'credit_score',
                                  'person_education']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['person_gender', 'person_home_ownership',
                                  'loan_intent',
                                  'previous_loan_defaults_on_file'])])

Train Test & Split

In [21]:
from sklearn.model_selection import train_test_split
x_train, x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [22]:
x_train.shape,x_test.shape

((36000, 13), (9000, 13))

# **Multiple Models Development**

In [23]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter=1000),
    "Random Forest" : RandomForestClassifier(n_estimators=20),
    "Bagging" : BaggingClassifier(n_estimators=200),
    "Extra Trees" : ExtraTreesClassifier(n_estimators=200,random_state=42)
}

**Models Evaluation & Comparative**

In [24]:
results = {}

for name, model in models.items():
  pipe = Pipeline([
      ("preprocessing", preprocessor),
      ("model", model)
  ])

  pipe.fit(x_train,y_train)
  y_pred = pipe.predict(x_test)

  acc = accuracy_score(y_test,y_pred)

  print("="*60)
  print(name)
  print("Accuracy",acc)
  print("Confusion Matrix:\n",confusion_matrix(y_test,y_pred))
  print("Classification Report:\n",classification_report(y_test,y_pred))

  results[name] = acc

Logistic Regression
Accuracy 0.8993333333333333
Confusion Matrix:
 [[6598  402]
 [ 504 1496]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.94      0.94      7000
           1       0.79      0.75      0.77      2000

    accuracy                           0.90      9000
   macro avg       0.86      0.85      0.85      9000
weighted avg       0.90      0.90      0.90      9000

Random Forest
Accuracy 0.9272222222222222
Confusion Matrix:
 [[6823  177]
 [ 478 1522]]
Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.97      0.95      7000
           1       0.90      0.76      0.82      2000

    accuracy                           0.93      9000
   macro avg       0.92      0.87      0.89      9000
weighted avg       0.93      0.93      0.93      9000

Bagging
Accuracy 0.9321111111111111
Confusion Matrix:
 [[6796  204]
 [ 407 1593]]
Classification Report:
           

# **Best Model Selection & Perform**

In [25]:
best_model_name = max(results,key=results.get)
print("Best Model:",best_model_name)

Best Model: Bagging


**Train Final Model**

In [26]:
final_model = Pipeline([
    ("preprocessing",preprocessor),
    ("model",models[best_model_name])
])
final_model.fit(x_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['person_age',
                                                   'person_income',
                                                   'person_emp_exp',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'cb_person_cred_hist_length',
                                                   'credit_score',
                                                   'person_education']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['person_gender',
                                                   'person_home_ownership',
                                                   'loan_intent',
                                                   'previous_loan_defaults_on_file'])])),
                ('model', BaggingClassifier(n_estimators=200))])

# **Prediction on Realtime Data**

In [29]:
new_person = pd.DataFrame([
    {
    "person_age": 26,
    "person_gender": "male",
    "person_education": 2,  # Bachelor
    "person_income": 85000,
    "person_emp_exp": 3,
    "person_home_ownership": "RENT",
    "loan_amnt": 15000,
    "loan_intent": "PERSONAL",
    "loan_int_rate": 12.5,
    "loan_percent_income": 0.2,
    "cb_person_cred_hist_length": 4,
    "credit_score": 690,
    "previous_loan_defaults_on_file": "No" # Changed from 0 to "No"
    }
])

prediction = final_model.predict(new_person)

if prediction[0] == 1:
    print("Loan Approved")
else:
    print("Loan Rejected")

Loan Rejected


In [30]:
# Human-readable data save karo (UI dropdowns ke liye)
df.to_csv('clean_loan.csv', index=False)

In [33]:
import joblib

# Trained pipeline save karo
joblib.dump(final_model, 'loan_pipeline.pkl')

# Download karo
from google.colab import files
files.download('loan_pipeline.pkl')
# files.download('clean_loan.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [32]:
import sklearn
print(sklearn.__version__)

1.6.1
